In [1]:
!pip install tensorflow pandas numpy scikit-learn matplotlib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, re, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
print('TF version:', tf.__version__)

TF version: 2.20.0


In [ ]:
# Hyperparameters
MAX_VOCAB  = 20000
MAX_LEN    = 200
EMBED_DIM  = 128
BATCH_SIZE = 128
EPOCHS     = 10

In [ ]:
# Load dataset and clean raw review text
def clean(text):
    text = re.sub(r'<[^>]+>', ' ', text)       # remove HTML tags
    text = text.lower()                         # lowercase
    text = re.sub(r'[^a-z\s]', '', text)       # remove punctuation and digits
    text = re.sub(r'\s+', ' ', text).strip()   # normalize whitespace
    return text

cwd = os.getcwd()
data_path = os.path.abspath(os.path.join(cwd, '..', '..', 'model', 'IMDB_Dataset.csv'))
df = pd.read_csv(data_path)
df['review'] = df['review'].apply(clean)
y = df['sentiment'].map({'positive': 1, 'negative': 0}).values
texts = df['review'].values
print(f'Loaded {len(df)} reviews')
df.head()

In [8]:
# Split data, tokenize vocabulary, and pad sequences to fixed length
X_train_t, X_test_t, y_train, y_test = train_test_split(
    texts, y, test_size=0.2, random_state=42, stratify=y
)

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_t)

X_train = pad_sequences(tokenizer.texts_to_sequences(X_train_t), maxlen=MAX_LEN, padding='post', truncating='post')
X_test  = pad_sequences(tokenizer.texts_to_sequences(X_test_t),  maxlen=MAX_LEN, padding='post', truncating='post')
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (40000, 200), Test: (10000, 200)


In [ ]:
# Define LSTM model architecture
model = Sequential([
    Embedding(MAX_VOCAB, EMBED_DIM, input_length=MAX_LEN),
    Dropout(0.2),
    LSTM(64),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Train with early stopping and learning rate reduction on plateau
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
]
history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks
)

In [ ]:
# Evaluate model and save training curves, confusion matrix, classification report
cwd = os.getcwd()
root_dir = os.path.abspath(os.path.join(cwd, '..', '..'))
results_dir = os.path.join(root_dir, 'results', 'model2_lstm')
os.makedirs(results_dir, exist_ok=True)

# Training curves
plt.figure(figsize=(10, 4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss'); plt.legend()
plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title('Accuracy'); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'training_curves.png'), dpi=150)
plt.show()

# Predictions
y_pred = (model.predict(X_test, batch_size=256) >= 0.5).astype(int).flatten()
print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(4, 4))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix - LSTM')
plt.xlabel('Predicted'); plt.ylabel('Actual')
for (i, j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha='center', va='center', color='black')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'confusion_matrix.png'), dpi=150)
plt.close()

# Classification report
report = classification_report(y_test, y_pred, output_dict=True)
labels_order = ['0', '1', 'macro avg', 'weighted avg']
metrics = ['precision', 'recall', 'f1-score']
report_matrix = np.array([[report[l][m] for m in metrics] for l in labels_order if l in report], dtype=np.float32)
row_labels = [l for l in labels_order if l in report]
plt.figure(figsize=(6, 3.5))
plt.imshow(report_matrix, cmap='Greens', aspect='auto')
plt.title('Classification Report - LSTM')
plt.xticks(range(len(metrics)), metrics)
plt.yticks(range(len(row_labels)), row_labels)
for (i, j), v in np.ndenumerate(report_matrix):
    plt.text(j, i, f'{v:.2f}', ha='center', va='center', color='black')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'classification_report.png'), dpi=150)
plt.close()
print(f'Evaluation images saved to {results_dir}')

In [ ]:
# Save model weights and tokenizer to results folder
model.save(os.path.join(results_dir, 'lstm_model.keras'))
with open(os.path.join(results_dir, 'tokenizer.pkl'), 'wb') as f:
    pickle.dump({'tokenizer': tokenizer, 'max_len': MAX_LEN, 'max_vocab': MAX_VOCAB}, f)
print(f'Model saved to {results_dir}')